# Parquet Basics — 08: Compression Codecs Deep Dive

## What you will learn
Choosing the right compression codec saves storage costs and improves performance.
The "best" codec depends on your data, hardware, and access patterns.

In this notebook:
1. How compression works inside Parquet (page-level)
2. Full benchmark: snappy vs zstd vs gzip vs lz4 vs none vs brotli
3. CPU vs I/O trade-off — when more compression is slower
4. zstd compression levels — fine-tuning ratio vs speed
5. Data type impact — which columns compress best
6. Production recommendations by workload type


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

## Step 1 — How Compression Works in Parquet

Parquet compresses at the **page level** (default 1 MB pages) within each column chunk.
Each page is compressed independently using the chosen codec.

```
Column Chunk (revenue):
  Page 0: [999.99, 49.99, 349.99, ...]  → compressed with codec → stored bytes
  Page 1: [129.99, 599.99, ...]         → compressed with codec → stored bytes
  ...

Dictionary page (category): ["Electronics","Books",...] → compressed once
Data pages (category):       [0, 1, 2, 0, 1, ...]      → indices, compress very well
```

This is why low-cardinality string columns compress so much better than floats —
the dictionary page holds all unique values, data pages hold tiny integer indices.


In [ ]:
import glob as glib

codecs_to_test = ["snappy", "zstd", "gzip", "lz4", "none"]
try:
    # brotli available in some pyarrow versions
    import pyarrow.parquet as pq
    codecs_to_test.append("brotli")
except:
    pass

OUT = f"{DATA_DIR}/codecs"
results = {}

print(f"Testing {len(codecs_to_test)} compression codecs on {N:,} rows...")
print()

for codec in codecs_to_test:
    path = f"{OUT}/{codec}"
    try:
        # Write
        t0 = time.time()
        df.write.mode("overwrite").option("compression", codec).parquet(path)
        t_write = time.time() - t0

        # Size
        files   = glib.glob(f"{path}/*.parquet")
        size_mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024

        # Read + full scan
        times = []
        for _ in range(3):
            t0 = time.time()
            spark.read.parquet(path).agg(F.sum("revenue"), F.count("*")).collect()
            times.append(time.time() - t0)
        t_read = sorted(times)[1]

        results[codec] = {"write": t_write, "size": size_mb, "read": t_read}
        print(f"  {codec:<8} write={t_write:.2f}s  size={size_mb:.1f}MB  read={t_read:.3f}s")
    except Exception as e:
        print(f"  {codec:<8} ERROR: {e}")

In [ ]:
# Summary table
base_size = results["none"]["size"]
base_write = results["none"]["write"]

print(f"\n{'Codec':<10} {'Write':>8} {'vs none':>8} {'Size MB':>10} {'Compression':>12} {'Read':>8}")
print("-" * 62)
for codec, r in results.items():
    write_ratio = f"{r['write']/base_write:.2f}x"
    compress_ratio = f"{base_size/r['size']:.2f}x"
    print(f"  {codec:<8} {r['write']:>6.2f}s {write_ratio:>8} "
          f"{r['size']:>8.1f} MB {compress_ratio:>12} {r['read']:>6.3f}s")

print()
if "zstd" in results and "snappy" in results:
    zstd_vs_snappy = results["snappy"]["size"] / results["zstd"]["size"]
    print(f"zstd vs snappy: {zstd_vs_snappy:.2f}x smaller, "
          f"{results['zstd']['write']/results['snappy']['write']:.2f}x slower write")

## Step 2 — zstd Compression Levels

zstd supports levels 1-22:
- Level 1: fast, similar to snappy
- Level 3: default — good balance (used by Spark's `zstd` option)
- Level 9: high compression, slower
- Level 19+: very high compression, much slower — rarely worth it


In [ ]:
zstd_path = f"{DATA_DIR}/zstd_levels"
zstd_results = {}

for level in [1, 3, 6, 9, 15]:
    path = f"{zstd_path}/level_{level}"
    t0 = time.time()
    df.write.mode("overwrite") \
            .option("compression", "zstd") \
            .option("parquet.compression.codec.zstd.level", str(level)) \
            .parquet(path)
    t_write = time.time() - t0

    files = glib.glob(f"{path}/*.parquet")
    size_mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024
    zstd_results[level] = {"write": t_write, "size": size_mb}
    print(f"  zstd level {level:>2}: {size_mb:.1f} MB  write={t_write:.2f}s")

base = zstd_results[1]["size"]
print()
print(f"  Level 1  → baseline (fastest zstd)")
for level, r in zstd_results.items():
    if level > 1:
        savings = (base - r["size"])/base * 100
        print(f"  Level {level:>2} → {savings:.0f}% smaller than level 1, "
              f"{r['write']/zstd_results[1]['write']:.1f}x slower write")

## Step 3 — Compression by Data Type

Different data types compress very differently:
- **Low-cardinality strings** (category, country): excellent — dict encoding + compression
- **High-cardinality strings** (emails, UUIDs): poor — no dictionary benefit
- **Doubles/floats**: poor — near-random bit patterns
- **Integers with patterns**: good — delta encoding then compression
- **Booleans**: excellent — RLE + compression on bit-packed values


In [ ]:
type_df = spark.createDataFrame([
    (i,
     f"cat_{i % 6}",              # low cardinality string
     f"user_{i}@email.com",       # high cardinality string
     float(i) * 3.14159,          # float (pseudo-random pattern)
     i % 1000,                    # integer with pattern
     i,                           # sequential integer
     i % 2 == 0,                  # boolean
    ) for i in range(200_000)
], ["id","category","email","amount","code","seq_id","is_active"])

type_path = f"{DATA_DIR}/type_compression"
for codec in ["snappy", "zstd", "none"]:
    path = f"{type_path}/{codec}"
    type_df.write.mode("overwrite").option("compression",codec).parquet(path)

# Measure per-column compression via pyarrow
try:
    import pyarrow.parquet as pq
    for codec in ["none", "snappy", "zstd"]:
        path = f"{type_path}/{codec}"
        files = glib.glob(f"{path}/*.parquet")
        if files:
            meta = pq.ParquetFile(files[0]).metadata
            rg   = meta.row_group(0)
            print(f"\nCodec: {codec}")
            for ci in range(min(7, meta.num_columns)):
                col = rg.column(ci)
                kb  = col.total_compressed_size / 1024 if hasattr(col, 'total_compressed_size') else col.total_byte_size / 1024
                print(f"  {col.path_in_schema:<20} {kb:>8.1f} KB")
except ImportError:
    # Fallback: measure file sizes per column via schema
    print("Column-level analysis (overall file sizes):")
    for codec in ["none","snappy","zstd"]:
        path = f"{type_path}/{codec}"
        files = glib.glob(f"{path}/*.parquet")
        mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024
        print(f"  {codec:<8}: {mb:.1f} MB")

## Summary: Compression Codec Selection Guide

| Workload | Recommended | Reason |
|---|---|---|
| Hot data (frequent reads) | `snappy` | Fast decompression, good ratio |
| Cold/archival storage | `zstd` level 6+ | Best ratio, acceptable speed |
| Temp/intermediate files | `lz4` | Fastest, minimal CPU |
| Maximum compression | `zstd` level 9-15 | Best ratio when storage costs dominate |
| CPU-bound workloads | `none` | Zero decompression overhead |

### Quick decision tree
```
Is storage cost the primary concern?
  YES → zstd (level 3-6)
  NO  ↓
Is this a frequently queried hot table?
  YES → snappy (fast decompression)
  NO  ↓
Is this a temporary/intermediate file?
  YES → lz4 (fastest)
  NO  → snappy (safe default)
```

```python
# Storage-optimized
.option("compression", "zstd")
.option("parquet.compression.codec.zstd.level", "6")

# Performance-optimized
.option("compression", "snappy")   # default

# Maximum speed
.option("compression", "lz4")
```
